# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

   Please __include at least two test cases__ showing that your HA/NA classifier works. You can copy the `run_tests` helper from lecture 9, or use `assert` statements - either is fine. These test cases are part of the points for this question.


In [14]:
import re
import Bio.SeqIO
import Bio.Seq
seqreads = list(Bio.SeqIO.parse("barcodes_R1.fastq", format="fastq"))

# test cases - these look a bit different than what we did on Thursday. Rather than giving just None for a failed search, 
# I give [None, "misformatted"] to match the formatting of returning both the gene and the barcode
# these tests look at what would happen if the barcode was not 16 characters long, if the barcode had a Y, and if there were a typo in the middle sequence 
# (which does not report either HA nor NA), and if neither the end of HA nor NA is found
test_cases = [
    # (read, expected result, label saying which requirement this checks)
    ("AGGGTGTGGCGGTGGTGCGGCCGCCTATGGTGCACTATTATTTATCTATCGTG", ["NA", "ACCACCGCCACACCCT"], "positive: clean NA read"),
    ("AGGGTGTGGCGGTGGTGCGGCCGCCTATGGTGCATCATTATATGCAAATCCGG", ["HA", "ACCACCGCCACACCCT"], "positive: clean HA read"),
    ("AGTGTGGCGGTGGTGCGGCCGCCTATGGTGCACTATTATTTATCTATCGTG", ['NA', 'misformatted'], "barcode is too short"),
    ("AGGGTYTGGCGGTGGTGCGGCCGCCTATGGTGCACTATTATTTATCTATCGTG", ['NA', 'misformatted'], "barcode base includes characters other than A, C, T, G, N"),
    ("AGGGTGTGGCGGTGGTGCAGCCGCCTATGGTGCACTATTATTTATCTATCGTG", [None, 'misformatted'], "wrong middle sequence"),
    ("AGGGTGTGGCGGTGGTGCGGCCGCCTATGGTGCACTATTATTTATCTATCAAA", [None, "misformatted"], "neither HA nor NA"),
]

# run_tests function given last thursday
def run_tests(parser_function, test_cases):
    """Run a parser against a list of test cases and report the results.

    Args:
        parser_function: The function to test. It takes a sequence and returns a barcode
            string, or None.
        test_cases: A list of (sequence, expected result, label) tuples.

    Returns:
        The number of test cases that failed.
    """
    n_failed = 0

    for seq, expected, label in test_cases:
        result = parser_function(seq)
        if result == expected:
            print(f"PASS  {label}")
        else:
            n_failed += 1
            print(f"FAIL  {label}: {seq!r} gave {result!r}, expected {expected!r}")

    return f"{n_failed} tests failed"

# this function gives pre-made barcode sequence lengths, upstream NA and HA values (correct orientation), and a middle section sequence (correct orientation). 
# all of these values can be changed
def classify_HA_NA(seqread, bclen=16, upstream_NA = "CACGATAGATAAATAATAGTGCACCAT", upstream_HA="CCGGATTTGCATATAATGATGCACCAT", middle_section="AGGCGGCCGC"):
    # args, returns, and example are all updated. Example is from the last sequence in the seqreads object.
    """Check whether a gene is HA or NA and give the barcode paired to the gene classification.

    Args:
        seqread: Nucleotide sequence read (a Seq object or a string) matching
            in reverse orientation.
        bclen: Length of the barcode.
        upstream_NA: Sequence expected immediately upstream of the barcode in influenza virus NA gene (forward orientation).
        upstream_HA: Sequence expected immediately upstream of the barcode in influenza virus HA gene (forward orientation).

    Returns:
        ["HA" or "NA", <barcode>]
        or
        ["HA" or "NA", "misformatted"] if the gene sequence is found but the barcode is misformatted
        or
        [None, "misformatted"] if the read does not match the expected design.

    Example:
        >>> classify_HA_NA(Bio.Seq.Seq("CGTAGGATTGAATTAGGCGGCCGCCTATGGTGCACTATTATTTATCTATCGTG"), bclen=16)
        ['NA', 'CTAATTCAATCCTACG']
    """
    
    # this takes care of any input that may be in lower case and takes care of any str versus Bio.Seq.Seq
    seqread_upper=str(seqread).upper()
    # convert to Bio.Seq.Seq object, take the reverse complement, and then turn back to string. This puts it into 5'-[end of NA or HA]-AGGCGGCCGC-[16 X N barcode]-3' format
    seqread_original=seqread_upper = str(Bio.Seq.Seq(seqread_upper).reverse_complement())
    # search for HA pattern. Start with upstream_HA (doesn't need to start with thtat), then specified middle section, then find and name the barcode which needs to be 
    # an A, T, C, G or N and must be either 16 bp long or another specified barcode length
    HA_pattern=re.compile(f"{upstream_HA}{middle_section}(?P<barcode>[ACGTN]{{{bclen}}})")
    # same as above but for NA gene sequence
    NA_pattern=re.compile(f"{upstream_NA}{middle_section}(?P<barcode>[ACGTN]{{{bclen}}})")
    # This allows the barcode to contain other characters or be the wrong number of bp but specifies that the HA gene was found and that the barcode was wrong
    barcode_misformatted_HA=re.compile(f"{upstream_HA}{middle_section}")
    # This allows the barcode to contain other characters or be the wrong number of bp but specifies that the NA gene was found and that the barcode was wrong
    barcode_misformatted_NA=re.compile(f"{upstream_NA}{middle_section}")

    # actually searching the specified input sequence for HA/NA gene ends where the barcode is correct or misformatted
    match_HA = HA_pattern.search(seqread_original)
    match_NA = NA_pattern.search(seqread_original)
    match_misformatted_NA=barcode_misformatted_NA.search(seqread_original)
    match_misformatted_HA=barcode_misformatted_HA.search(seqread_original)
    
    # the output of this a list that specifies the gene (HA or NA) and either the barcode or a string that specifies that the barcode was misformatted
    # If the middle sequence or the gene sequence was misformatted, the output will be the following list: [None, "misformatted"]
    if match_HA:
        dict_input=["HA", match_HA.group("barcode")]
    elif match_NA: 
        dict_input=["NA", match_NA.group("barcode")]
    elif match_misformatted_NA:
        dict_input=["NA", "misformatted"]
    elif match_misformatted_HA:
        dict_input=["HA", "misformatted"]
    else:
        dict_input=[None, "misformatted"]
    # return the output list
    return dict_input

# run the tests specified above
print("output of run_tests: ")
run_tests(classify_HA_NA, test_cases)

# initialize counts of accurately formatted NA or HA gene sequences with the correct middle sequence and the correctly formatted barcode
count_NA=0
count_HA=0
# initialize counts of accurately formatted NA or HA gene sequences with an incorrectly formatted barcode
count_NA_mis=0
count_HA_mis=0

# initialize dictionaries for the correctly formatted NA or HA barcodes
barcodes_HA = {}
barcodes_NA = {}

# loop through the seqread object that we read in above
for seqread in seqreads: 
    # run function that we defined above giving the list output
    gene, barcode = classify_HA_NA(seqread.seq)
    # if gene is HA and barcode is NOT incorrectly formatted, we increase the count of HA gene counts
    if gene == "HA" and barcode != "misformatted":
        count_HA += 1
        # we also create a dictionary of barcodes and their counts, where if the barcode isn't already in the dictionary, it's added with a count of 1
        barcodes_HA[barcode] = barcodes_HA.get(barcode, 0) + 1
    # same as above but for NA
    elif gene == "NA" and barcode != "misformatted":
        count_NA += 1
        barcodes_NA[barcode] = barcodes_NA.get(barcode, 0) + 1
    # if barcode IS misformatted, we only increase the counts of misformatted barcodes for HA gene counts. This answers question 2
    elif gene == "HA":
        count_HA_mis += 1
    # Same as above but for NA gene counts
    elif gene == "NA":
        count_NA_mis += 1

print("\n\n\nGene counts:")
print(f"There are {count_NA} NA gene counts")
print(f"There are {count_HA} HA gene counts")



output of run_tests: 
PASS  positive: clean NA read
PASS  positive: clean HA read
PASS  barcode is too short
PASS  barcode base includes characters other than A, C, T, G, N
PASS  wrong middle sequence
PASS  neither HA nor NA



Gene counts:
There are 3912 NA gene counts
There are 5250 HA gene counts


2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [15]:

# if we were only interested in answering this question, this would be how we would do that

# loop through the seqread object that we read in above after resetting
count_HA_mis=0
count_NA_mis=0
for seqread in seqreads: 
    # run function that we defined above giving the list output
    gene, barcode = classify_HA_NA(seqread.seq)
    # if gene is HA and barcode IS incorrectly formatted, we increase the count of HA gene counts with misformatted barcodes
    if gene == "HA" and barcode == "misformatted":
        count_HA_mis += 1
    # same as above but for NA
    elif gene == "NA" and barcode == "misformatted":
        count_NA_mis += 1

# please see above code for original implementation of this code: 
print(f"There are {count_HA_mis} HA genes with misformatted barcodes")
print(f"There are {count_NA_mis} NA genes with misformatted barcodes")

There are 66 HA genes with misformatted barcodes
There are 61 NA genes with misformatted barcodes


3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [16]:
# barcodes_HA.items() turns the dictionary into (barcode, count) tuples
# we then find the max value (not the first alphabetical barcode) with key=lambda item: item[1], which compares each tuple to the next element. 
# this technique can be found at this website: https://realpython.com/sort-python-dictionary/
max_HA_barcode, num_HA = max(barcodes_HA.items(), key=lambda item: item[1])
max_NA_barcode, num_NA = max(barcodes_NA.items(), key=lambda item: item[1])

# we return the full tuple, but we can break it up in in an f string
print(f"HA: the barcode {max_HA_barcode} has the most counts at {num_HA} counts")
print(f"NA: the barcode {max_NA_barcode} has the most counts at {num_NA} counts")

HA: the barcode CCCGACCCGACATTAA has the most counts at 155 counts
NA: the barcode ACCAGTTCTCCCCGGG has the most counts at 152 counts
